# <font color="steelblue">04. Tablas de datos con Pandas: leer, explorar, filtrar y limpiar</font>

*Unidad 1. Introducción* · Sesión teórica 4 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`, `salarios_redaccion.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv", "salarios_redaccion.csv"]
PREGUNTAS = ["q_P04_1.json", "q_P04_2.json", "q_P04_3.json", "q_P04_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Los datos de un periodista llegan casi siempre en forma de **tabla**: una hoja de cálculo del portal de transparencia, un CSV del INE, la base de datos de una encuesta. *Pandas* es la librería de Python para trabajar con esas tablas, que llama **dataframes**.

Antes de contar ninguna historia con una tabla hay que **conocerla**: cuántas filas tiene, qué significa cada columna, qué datos faltan. Este cuaderno es esa rutina.

**Qué necesitas saber antes**: los cuadernos 2 y 3.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Leer un fichero CSV como dataframe y ver su tamaño y sus columnas.
2. Distinguir los tipos de variable estadística: cuantitativas y cualitativas.
3. Seleccionar columnas y filtrar filas que cumplen una condición.
4. Detectar y tratar datos que faltan (`NaN`).

## <font color='steelblue'>Contenidos</font>

1. Leer y conocer la tabla
2. Tipos de variables
3. Seleccionar y filtrar
4. Datos que faltan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Leer y conocer la tabla</font>

**Qué vamos a hacer**: leer la encuesta de consumo de noticias y hacer las cuatro comprobaciones de rutina: tamaño, primeras filas, tipos y nombres de columna.

In [ ]:
encuesta = pd.read_csv("encuesta_medios.csv")
print("Filas y columnas:", encuesta.shape)
encuesta.head()

In [ ]:
encuesta.info()

**Cómo leer `info()`**: hay 1.200 filas. Casi todas las columnas tienen 1.200 valores no nulos, pero `minutos_noticias_dia` y `confianza_medios` tienen menos: **faltan respuestas**. `object` significa texto; `int64` y `float64`, números enteros y decimales.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P04_1.json")

## <font color='steelblue'>2. Tipos de variables</font>

En Estadística las variables se clasifican según **qué información recogen**, y de eso depende qué resumen y qué gráfico usar:

| Tipo | Subtipo | Ejemplo en la encuesta |
| :-- | :-- | :-- |
| **Cuantitativa** (números con sentido de cantidad) | Discreta (se cuenta) | edad en años, minutos al día |
| | Continua (admite decimales) | salario, share |
| **Cualitativa** (categorías) | Nominal (sin orden) | provincia, medio principal |
| | Ordinal (con orden) | nivel de estudios, confianza de 0 a 10 |
| | Binaria (dos valores) | comparte noticias sí/no |

⚠️ Una variable puede estar guardada con números y ser cualitativa: un código postal o un «1 = sí, 0 = no».

`describe()` resume las variables numéricas, y con `exclude="number"` las de texto:

In [ ]:
encuesta.describe().round(1)

In [ ]:
encuesta.describe(exclude="number")

**Cómo leerlo**: para las numéricas da recuento, media, desviación típica, mínimo, cuartiles y máximo (los verás en los cuadernos 8 a 10). Para las de texto, cuántos valores distintos hay (`unique`), cuál es el más frecuente (`top`) y cuántas veces aparece (`freq`).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P04_2.json")

## <font color='steelblue'>3. Seleccionar y filtrar</font>

Una columna se selecciona con su nombre entre corchetes. Varias columnas, con una **lista** de nombres (por eso hay dobles corchetes).

In [ ]:
encuesta[["edad", "medio_principal", "confianza_medios"]].head()

**Filtrar** es quedarse con las filas que cumplen una condición. Es la pregunta periodística más habitual: «¿y entre los jóvenes?».

In [ ]:
jovenes = encuesta[encuesta["edad"] < 30]
print("Menores de 30:", len(jovenes))
jovenes["medio_principal"].value_counts()

**Qué ha pasado**: entre los menores de 30 años, las **redes sociales** son el primer medio, cuando en el total de la encuesta eran el tercero. Esa diferencia ya es una historia.

Para combinar condiciones se usa `&` (y) y `|` (o), con cada condición entre paréntesis:

In [ ]:
filtro = (encuesta["edad"] < 30) & (encuesta["provincia"] == "Alicante")
print("Jóvenes de Alicante:", filtro.sum())

Y para ordenar, `sort_values()`. Estos son los cinco salarios más altos de la redacción simulada:

In [ ]:
salarios = pd.read_csv("salarios_redaccion.csv")
salarios.sort_values("salario_bruto_anual", ascending=False).head()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P04_3.json")

## <font color='steelblue'>4. Datos que faltan</font>

En una encuesta siempre hay quien no contesta. Pandas marca esos huecos con **`NaN`** (*Not a Number*). Antes de publicar una cifra hay que saber **cuántos faltan y por qué**.

In [ ]:
encuesta.isnull().sum()

**Cómo leerlo**: faltan 36 respuestas de minutos al día y 60 de confianza. Las funciones de Pandas, como `mean()`, **ignoran** los `NaN` automáticamente, así que la media de confianza se calcula sobre 1.140 personas, no sobre 1.200. Conviene decirlo en la noticia («entre quienes contestaron...»).

In [ ]:
print("Personas con dato de confianza:", encuesta["confianza_medios"].count())
print("Confianza media:", round(encuesta["confianza_medios"].mean(), 2))

completos = encuesta.dropna()
print("Filas sin ningún hueco:", len(completos))

`dropna()` elimina las filas con algún `NaN`. Úsalo con cuidado: si quienes no contestan son distintos de quienes sí lo hacen (por ejemplo, desconfían más), quitar los huecos **sesga** el resultado.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P04_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Lee `salarios_redaccion.csv` y muestra su tamaño, sus 5 primeras filas y el tipo de cada columna.

<details><summary>Ver solución</summary>

```python
salarios = pd.read_csv("salarios_redaccion.csv")
print(salarios.shape)
print(salarios.dtypes)
salarios.head()
```
</details>

### **Ejercicio 2** *(sección 2)*

Clasifica las columnas de `salarios` según su tipo estadístico (cuantitativa discreta o continua, cualitativa nominal u ordinal).

<details><summary>Ver idea clave</summary>

`anios_experiencia`: cuantitativa discreta; `salario_bruto_anual`: cuantitativa continua; `puesto`, `tipo_medio`, `sexo` y `contrato`: cualitativas nominales; `id`: identificador, no se analiza.
</details>

### **Ejercicio 3** *(sección 3)*

¿Cuántas personas de la redacción trabajan como autónomas? ¿Y cuántas son mujeres con contrato temporal?

<details><summary>Ver solución</summary>

```python
print((salarios["contrato"] == "Autónomo").sum())
print(((salarios["sexo"] == "Mujer") & (salarios["contrato"] == "Temporal")).sum())
```
</details>

### **Ejercicio 4** *(sección 3)*

Muestra los 10 salarios más bajos de los redactores y redactoras (`puesto == "Redactor/a"`), con su experiencia y tipo de contrato.

<details><summary>Ver solución</summary>

```python
red = salarios[salarios["puesto"] == "Redactor/a"]
red.sort_values("salario_bruto_anual")[["salario_bruto_anual", "anios_experiencia", "contrato"]].head(10)
```
</details>

### **Ejercicio 5** *(sección 4)*

En la encuesta, calcula qué porcentaje de personas no contestó a los minutos diarios de noticias. ¿Lo mencionarías en una noticia?

<details><summary>Ver solución</summary>

```python
round(encuesta["minutos_noticias_dia"].isnull().mean() * 100, 1)
```

Un 3 %. Es poco, pero la nota metodológica de la noticia debería decir sobre cuántas respuestas se calcula cada cifra.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Al recibir una tabla: **`shape`**, **`head()`**, **`info()`** y **`describe()`**.
- Las variables pueden ser **cuantitativas** (discretas o continuas) o **cualitativas** (nominales, ordinales o binarias); un número no siempre es una cantidad.
- Una columna se selecciona con `df["col"]`; varias, con `df[["a", "b"]]`; las filas, con `df[condición]`, combinando condiciones con `&` y `|`.
- Los datos que faltan aparecen como **`NaN`**: cuéntalos con `isnull().sum()` y di en la noticia sobre cuántos casos se calcula cada cifra.
- Eliminar filas incompletas con `dropna()` puede **sesgar** los resultados.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **DataFrame** | Tabla de Pandas: filas (casos) y columnas (variables). |
| **CSV** | Fichero de texto con valores separados por comas. |
| **`NaN`** | Marca de dato que falta. |
| **Filtrar** | Quedarse con las filas que cumplen una condición. |
| **Sesgo** | Error sistemático que desvía un resultado en una dirección. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| Todo sale en una sola columna al leer un CSV | El separador no es la coma | Usa `pd.read_csv(fichero, sep=";")` |
| `KeyError` al seleccionar una columna | El nombre no coincide exactamente | Revisa los nombres con `df.columns` |
| `ValueError: The truth value of a Series is ambiguous` | Has usado `and`/`or` con columnas | Usa `&` y `|` y pon cada condición entre paréntesis |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

Con esto termina la unidad 1. En el cuaderno 5 empieza la Estadística descriptiva: tablas de frecuencias.

**Siguiente cuaderno**: [05. Tablas de frecuencias: contar bien antes de publicar](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P05_tablas_de_frecuencias.ipynb)